# 07 - Grad-CAM (OPTIONAL EXTENSION)

**This notebook is optional. Skip it if you are short on time** - nothing in the results tables or
the paper comparison depends on it.

**What this notebook does**: overlays Grad-CAM heatmaps on test images for a trained MiniConvNet, to
see which regions drive each prediction. It is a qualitative sanity check, not a metric: it can show
that the model attends to lung tissue rather than to scanner borders or text annotations.

**What must already exist**: a MiniConvNet checkpoint from notebook 02
(`models/miniconvnet_<arch>_<split>.keras`).

**What "looks right"**: heat concentrated inside the lung field. Heat sitting on image corners, the
body outline or burnt-in text means the model is keying on acquisition artefacts, which is worth
reporting as a limitation.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data_utils import load_split, make_dataset
from src.train_utils import set_global_seeds, checkpoint_path, run_name_for

set_global_seeds(SEED)

ARCH = 'flatten'      # 'flatten' or 'gap'
SPLIT = 'faithful'    # 'faithful' or 'clean'
LAST_CONV_LAYER = 'conv4'   # final conv layer of the MiniConvNet backbone

run_name = run_name_for('miniconvnet', ARCH, SPLIT)
ckpt = checkpoint_path(run_name)
print('checkpoint:', ckpt, '| exists:', ckpt.exists())
if not ckpt.exists():
    raise FileNotFoundError(f'{ckpt} not found - run 02_train_miniconvnet.ipynb first.')

In [ ]:
model = tf.keras.models.load_model(ckpt)
model.summary()
print('\nconv layers:', [l.name for l in model.layers if isinstance(l, tf.keras.layers.Conv2D)])
assert LAST_CONV_LAYER in [l.name for l in model.layers], f'{LAST_CONV_LAYER} not in this model'

## 1. Grad-CAM implementation

Standard Grad-CAM: gradient of the predicted class score with respect to the last conv feature map,
globally average-pooled into per-channel weights, then a ReLU'd weighted sum of the feature maps.

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        model.inputs, [model.get_layer(last_conv_layer_name).output, model.output])

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + tf.keras.backend.epsilon())
    return heatmap.numpy(), int(pred_index), preds.numpy()[0]

In [ ]:
def overlay(img_uint8, heatmap, alpha=0.4):
    # plt.get_cmap, not matplotlib.cm.get_cmap (removed in matplotlib 3.9)
    hm = tf.image.resize(heatmap[..., None], IMG_SIZE).numpy()[..., 0]
    colored = plt.get_cmap('jet')(hm)[..., :3] * 255
    return np.clip(img_uint8 * (1 - alpha) + colored * alpha, 0, 255).astype('uint8')

## 2. Pick test images (one per class)

**Looks right**: 4 images, one per class, drawn from the test portion of the chosen split.

In [ ]:
sdf = load_split(SPLIT)
test_df = sdf[sdf['split'] == 'test']
picks = test_df.groupby('class', group_keys=False).head(2).reset_index(drop=True)
ds = make_dataset(picks, batch_size=len(picks), shuffle=False, augment=False)
images, labels = next(iter(ds))
print('selected', len(picks), 'images:')
print(picks[['class', 'filename']].to_string(index=False))

## 3. Heatmaps

Titles show true class -> predicted class with confidence. Mispredictions are the interesting ones:
look at where the heat sits when the model picks the wrong tumour subtype.

In [ ]:
n = len(picks)
fig, axes = plt.subplots(2, n, figsize=(3 * n, 6.5))
for i in range(n):
    img = images[i:i + 1]
    hm, pred_idx, probs = make_gradcam_heatmap(img, model, LAST_CONV_LAYER)
    raw = img[0].numpy().astype('uint8')

    axes[0, i].imshow(raw)
    axes[0, i].set_title(f'true: {CLASS_NAMES[int(labels[i])]}', fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(overlay(raw, hm))
    correct = int(labels[i]) == pred_idx
    axes[1, i].set_title(f'pred: {CLASS_NAMES[pred_idx]} ({probs[pred_idx]:.2f})'
                         f"{'' if correct else '  X'}", fontsize=8,
                         color='black' if correct else 'red')
    axes[1, i].axis('off')

fig.suptitle(f'Grad-CAM - {run_name} (layer {LAST_CONV_LAYER})')
fig.tight_layout()
out = FIGURES_DIR / f'gradcam_{run_name}.png'
fig.savefig(out, dpi=140, bbox_inches='tight')
plt.show()
print('saved', out)

## 4. Notes

Grad-CAM on this backbone is computed at 14x14 resolution and upsampled to 224x224, so the maps are
coarse - treat them as "which region", not "which pixel". They are a qualitative supplement to the
confusion-matrix analysis in notebook 06, which remains the primary explanation of model behaviour.